# r1 · Crisis exclusion, Elastic Net (block B3)

Do the VIX results in e3 depend on the GFC or COVID months? We drop those months from the evaluation only; nothing is refitted. Pairs come from `research_config.BLOCKS["B3"]`, and B3.2 (`en_vix_only` vs `en_no_macro`) is the main one.

Samples (`crisis_config.py`): full, ex-GFC, ex-COVID, ex-both. The windows are the NBER recessions, matched on the month being forecast (target_date). Because the first forecast is for March 2008, the GFC part of the evaluation sample is March 2008 to June 2009. The initial training window already contains the first three GFC months (December 2007 to February 2008), and later expanding windows keep all GFC observations. Crisis exclusion changes only which months are scored and no model is refitted.

Tests are the same as e3: Clark-West on log-RV MSE, DM on QLIKE, one-sided, Newey-West with the Andrews bandwidth. Dropping COVID leaves a three-month gap, which the lags ignore.

Output: `models/forecasting/robustness/results/r1_crisis_evaluation.csv`. `loss_ratio` is `a`'s loss over `b`'s, and `pct_change = (loss_ratio - 1) * 100`.


In [1]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
sys.path.insert(0, str(_checkout / "models" / "forecasting" / "robustness"))
from utils.paths import MODELS_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from research_config import BLOCKS, TICKERS
from crisis_config import SAMPLES, exclude

RESULTS_DIR = MODELS_DIR / "forecasting" / "robustness" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

block = BLOCKS["B3"]
df = pd.read_csv(FORECAST_DIR / "en_forecasts.csv", parse_dates=["date", "target_date"])
available = set(df["model"])
pairs = [p for p in block["pairs"] if p[1] in available and p[2] in available]
print(f"{len(pairs)} of {len(block['pairs'])} B3 pairs runnable")

3 of 3 B3 pairs runnable


In [2]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1


def one_sided_p(t):
    return float(1 - norm.cdf(t))

## Sample

Common sample: the (ETF, month) rows where all four variants have a forecast, as in e3. Each sample below removes the crisis months from it.


In [3]:
models = [m for m in block["models"] if m in available]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]
d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])

months = {s: exclude(d, s)["target_date"].nunique() for s in SAMPLES}
pd.Series(months, name="forecast months").to_frame().T

,full,ex_GFC,ex_COVID,ex_both
forecast months,214,198,211,195


## Tests by sample


In [4]:
LOSSES = ["QLIKE", "MSE_log", "MSE_level"]


def evaluate(sub, sample):
    # Same losses and tests as e3, on the months kept in `sub`.
    per_etf = sub.groupby(["model", "etf"])[LOSSES].mean().reset_index()
    pooled = sub.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
    loss_idx = pd.concat([per_etf, pooled]).set_index(["model", "etf"])
    n_months = sub["date"].nunique()
    rows = []
    for pair_id, a, b, _ in pairs:
        A = sub[sub["model"] == a].set_index(["date", "etf"])
        B = sub[sub["model"] == b].set_index(["date", "etf"])
        dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
        # Clark-West adjusted differential; positive favours a
        cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

        for etf in ["pooled"] + TICKERS:
            if etf == "pooled":
                dm_t, cw_t = dm_diff.groupby("date").mean(), cw.groupby("date").mean()
            else:
                dm_t, cw_t = dm_diff.xs(etf, level="etf"), cw.xs(etf, level="etf")
            for loss, test, t in [("QLIKE", "DM", -hac_t(dm_t)), ("MSE_log", "Clark-West", hac_t(cw_t))]:
                rows.append({
                    "sample": sample, "pair": pair_id, "a": a, "b": b, "ETF": etf,
                    "loss": loss, "test": test,
                    "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                    "gain_t": t, "p_value": one_sided_p(t), "n_months": n_months,
                })
    return pd.DataFrame(rows)


results = pd.concat([evaluate(exclude(d, s), s) for s in SAMPLES], ignore_index=True)
results["pct_change"] = (results["loss_ratio"] - 1) * 100   # negative: a has the lower loss
results.to_csv(RESULTS_DIR / "r1_crisis_evaluation.csv", index=False)

# The full sample must reproduce e3.
e3 = pd.read_csv(FORECAST_RESULTS_DIR / "e3_evaluation.csv")
check = results[results["sample"] == "full"].merge(
    e3, on=["pair", "ETF", "loss", "test"], suffixes=("", "_e3"))
assert len(check) == len(e3)
for col in ["loss_ratio", "gain_t", "p_value"]:
    assert np.allclose(check[col], check[col + "_e3"]), col
print("full sample matches e3_evaluation.csv")

full sample matches e3_evaluation.csv


## Pooled results

Each cell is `loss_ratio (p-value)`; a ratio below 1 means `a` has the lower loss.


In [5]:
def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sample(loss, test, etf="pooled"):
    t = results[results["ETF"].eq(etf) & results["loss"].eq(loss) & results["test"].eq(test)].copy()
    t["cell"] = [f"{r:.3f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="sample", values="cell")
    return table.loc[t["row"].drop_duplicates(), list(SAMPLES)].rename_axis(None)


for loss, test in [("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sample(loss, test))

MSE_log: ratio (p-value), Clark-West


sample,full,ex_GFC,ex_COVID,ex_both
B3.1: en_full_macro vs en_no_vix,0.882 (<0.001),0.901 (<0.001),0.883 (<0.001),0.906 (<0.001)
B3.2: en_vix_only vs en_no_macro,0.914 (<0.001),0.863 (<0.001),0.906 (<0.001),0.848 (<0.001)
B3.3: en_full_macro vs en_vix_only,1.027 (0.107),1.044 (0.122),1.012 (0.076),1.028 (0.024)


QLIKE: ratio (p-value), DM


sample,full,ex_GFC,ex_COVID,ex_both
B3.1: en_full_macro vs en_no_vix,0.855 (0.043),0.870 (0.090),0.911 (0.015),0.949 (0.016)
B3.2: en_vix_only vs en_no_macro,0.901 (0.002),0.878 (<0.001),0.910 (0.001),0.883 (<0.001)
B3.3: en_full_macro vs en_vix_only,1.140 (0.965),1.115 (0.952),1.102 (0.941),1.063 (0.953)


## Percentage changes, pooled

Percentage change in mean loss for `a` relative to `b`: `100 × (loss_ratio - 1)`. Negative values mean lower loss for `a`.


In [6]:
pct = (
    results[results["ETF"].eq("pooled") & results["test"].isin(["Clark-West", "DM"])]
    .assign(row=lambda t: t["pair"] + ": " + t["a"] + " vs " + t["b"],
            col=lambda t: t["loss"].map({"MSE_log": "Log MSE", "QLIKE": "QLIKE"}))
    .pivot_table(index=["row", "col"], columns="sample", values="pct_change")
    [list(SAMPLES)]
)
pct.index.names = ["", ""]
pct.map(lambda value: f"{value:+.2f}%")

sample                                         full   ex_GFC ex_COVID  ex_both
                                                                              
B3.1: en_full_macro vs en_no_vix   Log MSE  -11.83%   -9.90%  -11.71%   -9.44%
                                   QLIKE    -14.51%  -12.96%   -8.90%   -5.08%
B3.2: en_vix_only vs en_no_macro   Log MSE   -8.59%  -13.73%   -9.37%  -15.22%
                                   QLIKE     -9.95%  -12.16%   -9.03%  -11.67%
B3.3: en_full_macro vs en_vix_only Log MSE   +2.71%   +4.38%   +1.18%   +2.78%
                                   QLIKE    +14.01%  +11.47%  +10.22%   +6.33%

## B3.2 by sector (exploratory)

Log-MSE loss ratio (Clark-West p-value) for `en_vix_only` vs `en_no_macro`, by sector ETF. One test per sector with no multiple-testing correction, so a single sector's p-value is weak evidence.


In [7]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}

t = results[results["pair"].eq("B3.2") & results["loss"].eq("MSE_log")].copy()
t["cell"] = [f"{r:.3f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
table = t.pivot(index="ETF", columns="sample", values="cell").loc[TICKERS + ["pooled"], list(SAMPLES)]
table.index = [f"{e} {SECTORS[e]}" for e in TICKERS] + ["Pooled"]
table.rename_axis(None)

sample,full,ex_GFC,ex_COVID,ex_both
XLB Materials,0.886 (<0.001),0.836 (<0.001),0.875 (<0.001),0.818 (<0.001)
XLE Energy,0.990 (0.041),0.979 (0.014),0.990 (0.046),0.978 (0.016)
XLF Financials,0.848 (<0.001),0.816 (<0.001),0.840 (<0.001),0.803 (<0.001)
XLI Industrials,0.866 (<0.001),0.773 (<0.001),0.843 (<0.001),0.735 (<0.001)
XLY Cons. Disc.,1.009 (<0.001),0.937 (<0.001),1.016 (<0.001),0.936 (<0.001)
Pooled,0.914 (<0.001),0.863 (<0.001),0.906 (<0.001),0.848 (<0.001)
